# Script for comparing different CNN models 

In [3]:
import os.path as op
import mne 
import os
from termcolor import colored
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,mean_absolute_error,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle
import tensorflow as tf

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 


from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, Conv1DTranspose, MaxPooling1D, Flatten, Dense, Input, Dropout,BatchNormalization  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

mne.set_log_level("CRITICAL")

# Loading in Data

In [4]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration_1205.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration_1', 'Duration_2']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

features_all = features_all.rename(columns={
    'Duration_1': 'Duration_corr',
    'Duration_2': 'Duration_zygo'
})

features_all_store = features_all

x = np.isnan(features_all['Duration_zygo']) 
indices = np.where(x)[0]
features_all = features_all.drop(indices)

In [ ]:
np.shape(features_all_store)
 

# Functions

In [5]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?
    model.add(MaxPooling1D(pool_size=1)) 

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [6]:
# single head CNN model for duration 
def CNN_model_duration(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=3))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add another max pooling layer

    model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(1, activation='linear'))


    return model  # Return the compiled model

In [7]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    x = Conv1D(128, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)


    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


 
    shared = Dense(128, activation='relu')(x)
    shared = Dropout(0.5)(shared)

    # Count-specific hidden layers
    count_branch = Dense(32, activation='relu')(shared)
    count_branch = Dropout(0.2)(count_branch)

    count_output = Dense(
        num_classes,
        activation='softmax',
        name='count_output'
    )(count_branch)

    duration_branch = Dense(32, activation='relu')(shared)
    duration_branch = Dropout(0.2)(duration_branch) # try increasing drop out for more regularization? (increase if overfitting)

    duration_output = Dense(
        num_classes,
        activation='linear',
        name='duration_output'
    )(duration_branch)


    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [8]:
# single head CNN model for number of contractions  
def CNN_model_contraction_multichan(input_shape, num_classes,feature_num):
    global epoch_len
    
    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(16, kernel_size=3, activation='relu', input_shape=input_shape, kernel_regularizer=l2(0.0001))(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer
    x = Conv1D(32, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)



    x = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.7)(x)

    x = Flatten()(x)

    # two heads for each label 
    # output head for Zygo
    out_zygo = Dense(num_classes, activation='softmax', name="zygo_output", kernel_regularizer=l2(0.001))(x)

    # output head for Corr
    out_corr = Dense(num_classes, activation='softmax', name="corr_output", kernel_regularizer=l2(0.001))(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])
    
    return model 

In [9]:
def CNN_model_duration_multichan(input_shape, num_classes, feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(32, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(64, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)

    x = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.4)(x)

    # output heads
    out_zygo = Dense(1, activation='relu', name="zygo_output")(x)
    out_corr = Dense(1, activation='relu', name="corr_output")(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])

    return model

In [2]:
def CNN_model_fourhead_multichan(input_shape, num_classes, feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # --- Shared CNN backbone ---
    x = Conv1D(32, kernel_size=3, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)
    shared = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    shared = Dropout(0.5)(shared)

    # zygo branch 
    zygo_branch = Dense(32, activation='relu')(shared)
    zygo_branch = Dropout(0.3)(zygo_branch)
    zygo_count_output    = Dense(num_classes, activation='softmax', name='zygo_count_output')(zygo_branch)
    zygo_duration_output = Dense(1, activation='linear', name='zygo_duration_output')(zygo_branch)

    # corr branch 
    corr_branch = Dense(32, activation='relu')(shared)
    corr_branch = Dropout(0.3)(corr_branch)
    corr_count_output    = Dense(num_classes, activation='softmax', name='corr_count_output')(corr_branch)
    corr_duration_output = Dense(1, activation='linear', name='corr_duration_output')(corr_branch)

    # --- Model ---
    model = Model(
        inputs=inputs,
        outputs=[
            zygo_count_output,
            zygo_duration_output,
            corr_count_output,
            corr_duration_output
        ]
    )

    return model

# Single Channel Training 

## Single Head Count 

In [33]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_zygo = features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy()
y_corr = features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()

indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [54]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_contraction=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_contraction = CNN_model_contraction(input_shape, num_classes,feature_num)
    model_contraction.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

    early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_contraction.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_contraction.evaluate(X_val,y_val)
    cvScores_contraction.append(scores[1] * 100)

    k += 1 

# redefine fresh model 
#final_model = CNN_model_contraction(input_shape, num_classes, feature_num)
#final_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])

model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test), callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/10


2026-05-11 22:13:10.544342: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 11s 35ms/step - loss: 4.1292 - accuracy: 0.6215 - val_loss: 0.9431 - val_accuracy: 0.7335
Epoch 2/10
286/286 [==============================] - 9s 32ms/step - loss: 1.4294 - accuracy: 0.6541 - val_loss: 1.2729 - val_accuracy: 0.7326
Epoch 3/10
286/286 [==============================] - 9s 32ms/step - loss: 1.8921 - accuracy: 0.6625 - val_loss: 1.4804 - val_accuracy: 0.7230
Epoch 4/10
286/286 [==============================] - 9s 33ms/step - loss: 1.4454 - accuracy: 0.7053 - val_loss: 2.5357 - val_accuracy: 0.7260
Epoch 5/10
286/286 [==============================] - 10s 33ms/step - loss: 0.8958 - accuracy: 0.7650 - val_loss: 1.7685 - val_accuracy: 0.7011
Epoch 6/10
286/286 [==============================] - 9s 33ms/step - loss: 0.7039 - accuracy: 0.8046 - val_loss: 2.4499 - val_accuracy: 0.6573
Epoch 7/10
286/286 [==============================] - 9s 33ms/step - loss: 0.5822 - accuracy: 0.8306 - val_loss: 2.5158 - val_accuracy: 0.6818
Epoch 8/

2026-05-11 22:14:47.925629: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 10s 33ms/step - loss: 6.9246 - accuracy: 0.6140 - val_loss: 1.3334 - val_accuracy: 0.7225
Epoch 2/10
286/286 [==============================] - 9s 32ms/step - loss: 1.9303 - accuracy: 0.6425 - val_loss: 2.0921 - val_accuracy: 0.7230
Epoch 3/10
286/286 [==============================] - 9s 32ms/step - loss: 3.1074 - accuracy: 0.6358 - val_loss: 2.5504 - val_accuracy: 0.7204
Epoch 4/10
286/286 [==============================] - 9s 32ms/step - loss: 1.9912 - accuracy: 0.6874 - val_loss: 2.0447 - val_accuracy: 0.6543
Epoch 5/10
286/286 [==============================] - 9s 33ms/step - loss: 1.3465 - accuracy: 0.7347 - val_loss: 2.8030 - val_accuracy: 0.7151
Epoch 6/10
286/286 [==============================] - 9s 32ms/step - loss: 1.2348 - accuracy: 0.7582 - val_loss: 3.2655 - val_accuracy: 0.6490
Epoch 7/10
286/286 [==============================] - 9s 32ms/step - loss: 1.1865 - accuracy: 0.7760 - val_loss: 3.7712 - val_accuracy: 0.6722
Epoch 8/1

...

286/286 [==============================] - 12s 40ms/step - loss: 2.3873 - accuracy: 0.6598 - val_loss: 0.8260 - val_accuracy: 0.7322
Epoch 2/10
286/286 [==============================] - 9s 32ms/step - loss: 0.9701 - accuracy: 0.6987 - val_loss: 0.8744 - val_accuracy: 0.7287
Epoch 3/10
286/286 [==============================] - 10s 35ms/step - loss: 1.1498 - accuracy: 0.6915 - val_loss: 1.0394 - val_accuracy: 0.7234
Epoch 4/10
286/286 [==============================] - 9s 32ms/step - loss: 0.9871 - accuracy: 0.7444 - val_loss: 1.2004 - val_accuracy: 0.7072
Epoch 5/10
286/286 [==============================] - 9s 32ms/step - loss: 0.7353 - accuracy: 0.7878 - val_loss: 2.0580 - val_accuracy: 0.5947
Epoch 6/10
286/286 [==============================] - 9s 32ms/step - loss: 0.6650 - accuracy: 0.8271 - val_loss: 1.7881 - val_accuracy: 0.6858
Epoch 7/10
286/286 [==============================] - 9s 33ms/step - loss: 0.4668 - accuracy: 0.8662 - val_loss: 2.5085 - val_accuracy: 0.6486
Epoch 8/

2026-05-11 22:18:00.923683: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 10s 32ms/step - loss: 2.6946 - accuracy: 0.6579 - val_loss: 0.9091 - val_accuracy: 0.7212
Epoch 2/10
286/286 [==============================] - 10s 34ms/step - loss: 1.1464 - accuracy: 0.6795 - val_loss: 1.0150 - val_accuracy: 0.7204
Epoch 3/10
286/286 [==============================] - 9s 31ms/step - loss: 1.3171 - accuracy: 0.6776 - val_loss: 1.2333 - val_accuracy: 0.7077
Epoch 4/10
286/286 [==============================] - 9s 33ms/step - loss: 1.1398 - accuracy: 0.7170 - val_loss: 1.6675 - val_accuracy: 0.6884
Epoch 5/10
286/286 [==============================] - 9s 32ms/step - loss: 0.9200 - accuracy: 0.7502 - val_loss: 2.2135 - val_accuracy: 0.7011
Epoch 6/10
286/286 [==============================] - 9s 32ms/step - loss: 0.8276 - accuracy: 0.7902 - val_loss: 2.3437 - val_accuracy: 0.6779
Epoch 7/10
286/286 [==============================] - 9s 32ms/step - loss: 0.6326 - accuracy: 0.8282 - val_loss: 2.5871 - val_accuracy: 0.6311
Epoch 8/

2026-05-11 22:19:35.562656: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 10s 32ms/step - loss: 2.8643 - accuracy: 0.6598 - val_loss: 0.8578 - val_accuracy: 0.7259
Epoch 2/10
286/286 [==============================] - 9s 31ms/step - loss: 0.9224 - accuracy: 0.6979 - val_loss: 0.8707 - val_accuracy: 0.7255
Epoch 3/10
286/286 [==============================] - 9s 32ms/step - loss: 1.0573 - accuracy: 0.6926 - val_loss: 1.0161 - val_accuracy: 0.7176
Epoch 4/10
286/286 [==============================] - 9s 32ms/step - loss: 1.2139 - accuracy: 0.6985 - val_loss: 1.7765 - val_accuracy: 0.7084
Epoch 5/10
286/286 [==============================] - 9s 33ms/step - loss: 0.9919 - accuracy: 0.7545 - val_loss: 1.6916 - val_accuracy: 0.6686
Epoch 6/10
286/286 [==============================] - 9s 33ms/step - loss: 0.6459 - accuracy: 0.8127 - val_loss: 2.2040 - val_accuracy: 0.6594
Epoch 7/10
286/286 [==============================] - 9s 32ms/step - loss: 0.5277 - accuracy: 0.8516 - val_loss: 2.4269 - val_accuracy: 0.6734
Epoch 8/1

In [55]:
# cross validation results 
avgScores = np.mean(cvScores_contraction)
stdScores = np.std(cvScores_contraction)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

# full training results (test data not seen during cross val)
y_pred_train = model_contraction.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model_contraction.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   


# Calculate accuracy
accuracy_training = accuracy_score(y_train_full, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train_full, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Model scores---------------")
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test) 

Average KFold Cross Validation Score: 64.49579954147339
Standard Deviation KFold Cross Validation Score: 7.039713588931853
90/90 [==============================] - 1s 6ms/step
Model scores---------------
Training Accuracy : 0.9424019607843137
Test Accuracy : 0.8609943977591037
Training F1 Score : 0.9377815556332582
Test F1 Score : 0.8417281866980321


# Count by Muscle Groups 

In [ ]:
zygo_ind_train = np.where(idx_train < 7140)
corr_ind_train = np.where(idx_train >= 7140)  
zygo_ind_test = np.where(idx_test < 7140)
corr_ind_test = np.where(idx_test >= 7140)  

In [ ]:
# remake splits (i think this is wrong)
X_train_full_corr = X[corr_ind_train]
X_train_full_zygo = X[zygo_ind_train]

X_test_corr = X[corr_ind_test]
X_test_zygo = X[zygo_ind_test

y_train_full_corr = y[corr_ind_train]
y_train_full_zygo = y[zygo_ind_train]

y_test_corr = y[corr_ind_test]
y_test_zygo = y[zygo_ind_test]


# full training results (test data not seen during cross val)
y_pred_train_corr = model_contraction.predict(X_train_full_corr)  
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)   

y_pred_train_zygo = model_contraction.predict(X_train_full_zygo)  
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)   

# Predict on test data
y_pred_test_zygo = model_contraction.predict(X_test_zygo)   
y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)   

y_pred_test_corr= model_contraction.predict(X_test_corr)   
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)   

# Calculate accuracy
accuracy_training_corr = accuracy_score(y_train_full_corr, y_pred_train_corr)   
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)  

accuracy_training_zygo = accuracy_score(y_train_full_zygo, y_pred_train_zygo)   
accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)  

# Calculate F1 score
f1_training_corr = f1_score(y_train_full_corr, y_pred_train_corr, average='weighted')  
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')  

f1_training_zygo = f1_score(y_train_full_zygo, y_pred_train_zygo, average='weighted')  
f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')  


# Print accuracy and F1 score
print("Corru Scores------------------------------")  
print("Training Accuracy :", accuracy_training_corr) 
print("Test Accuracy :", accuracy_test_corr)  
print("Training F1 Score :", f1_training_corr)  
print("Test F1 Score :", f1_test_corr) 

print("Zygo Scores------------------------------")  
print("Training Accuracy :", accuracy_training_zygo)  
print("Test Accuracy :", accuracy_test_zygo)  
print("Training F1 Score :", f1_training_zygo)   
print("Test F1 Score :", f1_test_zygo) 

In [53]:
corr_mask = muscle_test == "Corr"
zygo_mask = muscle_test == "Zygo"

print("\nCorr Results-------------------")
#print("Accuracy Train:", accuracy_score(y_train_full[corr_mask], y_pred_train[corr_mask]))
print("Accuracy Test:", accuracy_score(y_test[corr_mask], y_pred_test[corr_mask]))

#print("F1 Train:", f1_score(y_train_full[corr_mask], y_pred_train[corr_mask]))
print("F1 Test:", f1_score(y_test[corr_mask], y_pred_test[corr_mask], average='weighted')  )

print("\nZygo Results-------------------")
#print("Accuracy Train:", accuracy_score(y_train_full[zygo_mask], y_pred_train[zygo_mask]))
print("Accuracy Test:", accuracy_score(y_test[zygo_mask], y_pred_test[zygo_mask])  )

#print("F1 Train:", f1_score(y_train_full[zygo_mask], y_pred_train[zygo_mask]))
print("F1 Test:", f1_score(y_test[zygo_mask], y_pred_test[zygo_mask], average='weighted')  )



Corr Results-------------------
Accuracy Test: 0.9032258064516129
F1 Test: 0.8895109493957513

Zygo Results-------------------
Accuracy Test: 0.8855869242199108
F1 Test: 0.8727681411763692


# Single Head Duration

In [10]:
# Define CNN model inputs
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all

muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = (np.concatenate((X_zygo, X_corr), axis=0))
y = (np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]]))     


indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [14]:
 KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_duration=[]
epoch_num = 100 
k = 1
 
for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]


    model_duration = CNN_model_duration(input_shape, num_classes,feature_num)
    model_duration.compile(optimizer='adam', loss='mae', metrics=['mae'])  

    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model_duration.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_duration.evaluate(X_val,y_val)
    cvScores_duration.append(scores[1])

    k += 1 kf =
    

model_history_duration = model_duration.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test), callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/100


2026-05-12 07:23:04.497442: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 26s 85ms/step - loss: 10.0085 - mae: 10.0085 - val_loss: 4.3119 - val_mae: 4.3119
Epoch 2/100
286/286 [==============================] - 23s 81ms/step - loss: 5.7317 - mae: 5.7317 - val_loss: 3.7286 - val_mae: 3.7286
Epoch 3/100
286/286 [==============================] - 22s 76ms/step - loss: 4.7696 - mae: 4.7696 - val_loss: 3.4743 - val_mae: 3.4743
Epoch 4/100
286/286 [==============================] - 21s 73ms/step - loss: 4.7204 - mae: 4.7204 - val_loss: 3.3673 - val_mae: 3.3673
Epoch 5/100
286/286 [==============================] - 21s 74ms/step - loss: 4.0324 - mae: 4.0324 - val_loss: 3.5267 - val_mae: 3.5267
Epoch 6/100
286/286 [==============================] - 21s 75ms/step - loss: 3.5053 - mae: 3.5053 - val_loss: 3.2809 - val_mae: 3.2809
Epoch 7/100
286/286 [==============================] - 21s 74ms/step - loss: 3.3100 - mae: 3.3100 - val_loss: 3.3086 - val_mae: 3.3086
Epoch 8/100
286/286 [==============================] - 21s 74ms/s

2026-05-12 07:27:28.455355: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 22s 73ms/step - loss: 8.6905 - mae: 8.6905 - val_loss: 3.5201 - val_mae: 3.5201
Epoch 2/100
286/286 [==============================] - 20s 71ms/step - loss: 3.7606 - mae: 3.7606 - val_loss: 3.3489 - val_mae: 3.3489
Epoch 3/100
286/286 [==============================] - 21s 72ms/step - loss: 3.8804 - mae: 3.8804 - val_loss: 3.3793 - val_mae: 3.3793
Epoch 4/100
286/286 [==============================] - 21s 74ms/step - loss: 3.8320 - mae: 3.8320 - val_loss: 3.5102 - val_mae: 3.5102
Epoch 5/100
286/286 [==============================] - 22s 77ms/step - loss: 3.7493 - mae: 3.7493 - val_loss: 3.3627 - val_mae: 3.3627
Epoch 6/100
286/286 [==============================] - 24s 83ms/step - loss: 3.8391 - mae: 3.8391 - val_loss: 3.3551 - val_mae: 3.3551
Epoch 7/100
286/286 [==============================] - 22s 77ms/step - loss: 3.5664 - mae: 3.5664 - val_loss: 3.7646 - val_mae: 3.7646
Epoch 8/100
90/90 [==============================] - 2s 21ms/step -

...

286/286 [==============================] - 25s 86ms/step - loss: 7.9694 - mae: 7.9694 - val_loss: 3.5743 - val_mae: 3.5743
Epoch 2/100
286/286 [==============================] - 24s 82ms/step - loss: 3.7031 - mae: 3.7031 - val_loss: 3.2966 - val_mae: 3.2966
Epoch 3/100
286/286 [==============================] - 24s 83ms/step - loss: 3.7702 - mae: 3.7702 - val_loss: 3.5995 - val_mae: 3.5995
Epoch 4/100
286/286 [==============================] - 24s 82ms/step - loss: 3.6102 - mae: 3.6102 - val_loss: 3.4825 - val_mae: 3.4825
Epoch 5/100
286/286 [==============================] - 24s 82ms/step - loss: 3.5199 - mae: 3.5199 - val_loss: 3.9625 - val_mae: 3.9625
Epoch 6/100
286/286 [==============================] - 24s 84ms/step - loss: 3.2864 - mae: 3.2864 - val_loss: 3.5628 - val_mae: 3.5628
Epoch 7/100
286/286 [==============================] - 31s 108ms/step - loss: 3.1937 - mae: 3.1937 - val_loss: 3.5613 - val_mae: 3.5613
Epoch 8/100
90/90 [==============================] - 3s 33ms/step 

2026-05-12 07:33:56.424627: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 764s 3s/step - loss: 15.3368 - mae: 15.3368 - val_loss: 4.4156 - val_mae: 4.4156
Epoch 2/100
286/286 [==============================] - 25s 88ms/step - loss: 7.6599 - mae: 7.6599 - val_loss: 3.8595 - val_mae: 3.8595
Epoch 3/100
286/286 [==============================] - 22s 78ms/step - loss: 5.2119 - mae: 5.2119 - val_loss: 3.6757 - val_mae: 3.6757
Epoch 4/100
286/286 [==============================] - 21s 72ms/step - loss: 4.8902 - mae: 4.8902 - val_loss: 3.7757 - val_mae: 3.7757
Epoch 5/100
286/286 [==============================] - 21s 73ms/step - loss: 4.9241 - mae: 4.9241 - val_loss: 3.8161 - val_mae: 3.8161
Epoch 6/100
286/286 [==============================] - 20s 69ms/step - loss: 4.1839 - mae: 4.1839 - val_loss: 3.5303 - val_mae: 3.5303
Epoch 7/100
286/286 [==============================] - 20s 71ms/step - loss: 3.9299 - mae: 3.9299 - val_loss: 3.4858 - val_mae: 3.4858
Epoch 8/100
286/286 [==============================] - 20s 70ms/st

2026-05-12 07:52:19.709827: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


286/286 [==============================] - 21s 72ms/step - loss: 9.6312 - mae: 9.6312 - val_loss: 3.9228 - val_mae: 3.9228
Epoch 2/100
286/286 [==============================] - 21s 73ms/step - loss: 5.1679 - mae: 5.1679 - val_loss: 3.6291 - val_mae: 3.6291
Epoch 3/100
286/286 [==============================] - 21s 74ms/step - loss: 3.8818 - mae: 3.8818 - val_loss: 3.6247 - val_mae: 3.6247
Epoch 4/100
286/286 [==============================] - 22s 76ms/step - loss: 5.8969 - mae: 5.8969 - val_loss: 5.6737 - val_mae: 5.6737
Epoch 5/100
286/286 [==============================] - 22s 76ms/step - loss: 5.6665 - mae: 5.6665 - val_loss: 4.3927 - val_mae: 4.3927
Epoch 6/100
286/286 [==============================] - 21s 73ms/step - loss: 3.5869 - mae: 3.5869 - val_loss: 3.4843 - val_mae: 3.4843
Epoch 7/100
286/286 [==============================] - 20s 70ms/step - loss: 3.4148 - mae: 3.4148 - val_loss: 3.5106 - val_mae: 3.5106
Epoch 8/100
286/286 [==============================] - 20s 72ms/ste

In [25]:
# cross validation results 
cvScores_duration = np.array(cvScores_duration)

avgScores = np.mean(cvScores_duration)
stdScores = np.std(cvScores_duration)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
 
baseline = np.mean(y_train_full)
mae_baseline = np.mean(np.abs(y_train_full - baseline))

# full training results (test data not seen during cross val)
y_pred_train_dur = model_duration.predict(X_train_full) 

# Predict on test data
y_pred_test_dur = model_duration.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full, y_pred_train_dur)   
mae_test_dur= mean_absolute_error(y_test, y_pred_test_dur)  
  
# Print accuracy and F1 score
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)

corr_mask = muscle_test == "Corr"
zygo_mask = muscle_test == "Zygo"

print("\nCorr Results-------------------")
print("MAE:", mean_absolute_error(y_test[corr_mask], y_pred_test_dur[corr_mask]))

print("\nZygo Results-------------------")
print("MAE:", mean_absolute_error(y_test[zygo_mask], y_pred_test_dur[zygo_mask]))
 

Average KFold Cross Validation Score: 3.1284050941467285
Standard Deviation KFold Cross Validation Score: 0.06855012514803999
90/90 [==============================] - 2s 17ms/step
Baseline MAE: 5.9552841687640035
Training MAE : 3.0984962507436227
Test MAE : 3.131378946321174

Corr Results-------------------
MAE: 2.93179616004585

Zygo Results-------------------
MAE: 3.5202189501141636


# Two Head 

In [26]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all

muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test, muscle_train, muscle_test = train_test_split(indices, muscle_groups,test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [58]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 

cvScores_dur =[]
cvScores_contr =[]
cvScores = [] 

epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_twohead = CNN_model_twohead(input_shape, num_classes,feature_num) #CNN_model_regression if treating both as continuous
    model_twohead.compile(optimizer='adam',loss=['sparse_categorical_crossentropy', 'mae'],metrics=['accuracy', 'mae'] ) #think about metric 
    model_history_kfold = model_twohead.fit(X_train,[y_train[:,0], y_train[:,1]], 
                                            validation_data=(X_val,[y_val[:,0], y_val[:,1]]), 
                                            epochs=epoch_num)
    
    scores = model_twohead.evaluate(X_val,[y_val[:,0], y_val[:,1]])
    
    metrics = dict(zip(model_twohead.metrics_names, scores))

    cvScores_dur.append(metrics['duration_output_mae'])
    cvScores_contr.append(metrics['count_output_accuracy'] * 100)


    cvScores.append(scores)
 
    
    

    k += 1 
    

model_history_twohead = model_twohead.fit(X_train_full, [y_train_full[:,0], y_train_full[:,1]], epochs=epoch_num, validation_data=(X_test, [y_test[:,0], y_test[:,1]]), callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/10


In [ ]:
# comparison for two head model 
# cross validation results 
avgScores_dur = np.mean(cvScores,axis=0)[3]
stdScores_dur = np.std(cvScores,axis=0)[3]

avgScores_contr= np.mean(cvScores,axis=0)[6]
stdScores_contr = np.std(cvScores,axis=0)[6]
 

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model_twohead.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model_twohead.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   

# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full[:,0], y_pred_train_contraction)   
accuracy_test_contraction = accuracy_score(y_test[:,0], y_pred_test_contraction)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full[:,0], y_pred_train_contraction, average='weighted')  
f1_test_contraction = f1_score(y_test[:,0], y_pred_test_contraction, average='weighted')  

# MAE 
baseline = np.mean(y_train_full[:,1])
mae_baseline = np.mean(np.abs(y_train_full[:,1] - baseline))

y_pred_train_dur = model_twohead.predict(X_train_full) 
y_pred_test_dur = model_twohead.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full[:,1], y_pred_train)   
mae_test_dur= mean_absolute_error(y_test[:,1], y_pred_test)  
  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)   
print("----------------------") 
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)


## Two Channel 

Single Head Count 

In [10]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


muscle_groups = features_all_temp["True_Muscle_Activated"].repeat(2).reset_index(drop=True)

X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]
 

In [28]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 100 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model = CNN_model_contraction_multichan(input_shape, num_classes,feature_num)
    model.compile(
    optimizer='adam',
    loss={
        'zygo_output': 'sparse_categorical_crossentropy',
        'corr_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'zygo_output': ['accuracy'],
        'corr_output': ['accuracy']
    }
)
    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]]), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
   
    scores = model.evaluate(X_val,[y_val[:, 0], y_val[:, 1]])
    for name, value in zip(model.metrics_names, scores):
        print(f"{name}: {value:.4f}")
    
    cvScores.append([scores[3]* 100,scores[4]* 100,( scores[3] + scores[4]) / 2 * 100]) 

    k += 1 
    

model_history = model.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

Fold: 1 ==================================================================
Epoch 1/100


2026-05-12 13:03:20.028354: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


143/143 [==============================] - 14s 78ms/step - loss: 2.3951 - zygo_output_loss: 1.1179 - corr_output_loss: 1.1920 - zygo_output_accuracy: 0.7971 - corr_output_accuracy: 0.7680 - val_loss: 2.0114 - val_zygo_output_loss: 0.8529 - val_corr_output_loss: 1.0621 - val_zygo_output_accuracy: 0.8556 - val_corr_output_accuracy: 0.8399
Epoch 2/100
143/143 [==============================] - 8s 55ms/step - loss: 1.4550 - zygo_output_loss: 0.6423 - corr_output_loss: 0.7079 - zygo_output_accuracy: 0.8407 - corr_output_accuracy: 0.8181 - val_loss: 1.4696 - val_zygo_output_loss: 0.6189 - val_corr_output_loss: 0.7372 - val_zygo_output_accuracy: 0.8644 - val_corr_output_accuracy: 0.8714
Epoch 3/100
143/143 [==============================] - 8s 55ms/step - loss: 1.2603 - zygo_output_loss: 0.5435 - corr_output_loss: 0.5959 - zygo_output_accuracy: 0.8496 - corr_output_accuracy: 0.8446 - val_loss: 1.3192 - val_zygo_output_loss: 0.5540 - val_corr_output_loss: 0.6388 - val_zygo_output_accuracy: 0.8

2026-05-12 13:05:11.020438: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


143/143 [==============================] - 12s 70ms/step - loss: 2.5045 - zygo_output_loss: 1.1774 - corr_output_loss: 1.2393 - zygo_output_accuracy: 0.8011 - corr_output_accuracy: 0.7634 - val_loss: 1.9702 - val_zygo_output_loss: 0.9732 - val_corr_output_loss: 0.8981 - val_zygo_output_accuracy: 0.8530 - val_corr_output_accuracy: 0.8346
Epoch 2/100
143/143 [==============================] - 8s 55ms/step - loss: 1.4189 - zygo_output_loss: 0.6249 - corr_output_loss: 0.6862 - zygo_output_accuracy: 0.8536 - corr_output_accuracy: 0.8282 - val_loss: 1.5117 - val_zygo_output_loss: 0.6691 - val_corr_output_loss: 0.7267 - val_zygo_output_accuracy: 0.8626 - val_corr_output_accuracy: 0.8513
Epoch 3/100
143/143 [==============================] - 9s 66ms/step - loss: 1.2800 - zygo_output_loss: 0.5648 - corr_output_loss: 0.5919 - zygo_output_accuracy: 0.8573 - corr_output_accuracy: 0.8409 - val_loss: 1.3631 - val_zygo_output_loss: 0.5967 - val_corr_output_loss: 0.6363 - val_zygo_output_accuracy: 0.8

...

143/143 [==============================] - 13s 76ms/step - loss: 2.9225 - zygo_output_loss: 1.2947 - corr_output_loss: 1.5420 - zygo_output_accuracy: 0.7646 - corr_output_accuracy: 0.7641 - val_loss: 2.0670 - val_zygo_output_loss: 0.9252 - val_corr_output_loss: 1.0473 - val_zygo_output_accuracy: 0.8634 - val_corr_output_accuracy: 0.8152
Epoch 2/100
143/143 [==============================] - 12s 81ms/step - loss: 1.4888 - zygo_output_loss: 0.6772 - corr_output_loss: 0.7067 - zygo_output_accuracy: 0.8361 - corr_output_accuracy: 0.8381 - val_loss: 1.3904 - val_zygo_output_loss: 0.6400 - val_corr_output_loss: 0.6365 - val_zygo_output_accuracy: 0.8538 - val_corr_output_accuracy: 0.8713
Epoch 3/100
143/143 [==============================] - 8s 55ms/step - loss: 1.2633 - zygo_output_loss: 0.5607 - corr_output_loss: 0.5831 - zygo_output_accuracy: 0.8519 - corr_output_accuracy: 0.8499 - val_loss: 1.2845 - val_zygo_output_loss: 0.5649 - val_corr_output_loss: 0.5946 - val_zygo_output_accuracy: 0.

2026-05-12 13:08:41.845350: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


143/143 [==============================] - 12s 71ms/step - loss: 2.5176 - zygo_output_loss: 1.1232 - corr_output_loss: 1.3084 - zygo_output_accuracy: 0.7860 - corr_output_accuracy: 0.7392 - val_loss: 2.0256 - val_zygo_output_loss: 0.9374 - val_corr_output_loss: 0.9902 - val_zygo_output_accuracy: 0.8573 - val_corr_output_accuracy: 0.8231
Epoch 2/100
143/143 [==============================] - 8s 58ms/step - loss: 1.4164 - zygo_output_loss: 0.6242 - corr_output_loss: 0.6844 - zygo_output_accuracy: 0.8464 - corr_output_accuracy: 0.8269 - val_loss: 1.3144 - val_zygo_output_loss: 0.5741 - val_corr_output_loss: 0.6211 - val_zygo_output_accuracy: 0.8590 - val_corr_output_accuracy: 0.8704
Epoch 3/100
143/143 [==============================] - 10s 72ms/step - loss: 1.2692 - zygo_output_loss: 0.5504 - corr_output_loss: 0.5955 - zygo_output_accuracy: 0.8571 - corr_output_accuracy: 0.8453 - val_loss: 1.2329 - val_zygo_output_loss: 0.5471 - val_corr_output_loss: 0.5539 - val_zygo_output_accuracy: 0.

2026-05-12 13:10:29.762205: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


143/143 [==============================] - 15s 89ms/step - loss: 2.8982 - zygo_output_loss: 1.3725 - corr_output_loss: 1.4396 - zygo_output_accuracy: 0.7632 - corr_output_accuracy: 0.7521 - val_loss: 1.9761 - val_zygo_output_loss: 0.9063 - val_corr_output_loss: 0.9737 - val_zygo_output_accuracy: 0.8625 - val_corr_output_accuracy: 0.8581
Epoch 2/100
143/143 [==============================] - 14s 98ms/step - loss: 1.4700 - zygo_output_loss: 0.6431 - corr_output_loss: 0.7193 - zygo_output_accuracy: 0.8405 - corr_output_accuracy: 0.8230 - val_loss: 1.3355 - val_zygo_output_loss: 0.5685 - val_corr_output_loss: 0.6484 - val_zygo_output_accuracy: 0.8538 - val_corr_output_accuracy: 0.8476
Epoch 3/100
143/143 [==============================] - 11s 78ms/step - loss: 1.2939 - zygo_output_loss: 0.5612 - corr_output_loss: 0.6076 - zygo_output_accuracy: 0.8479 - corr_output_accuracy: 0.8425 - val_loss: 1.2668 - val_zygo_output_loss: 0.5279 - val_corr_output_loss: 0.6061 - val_zygo_output_accuracy: 0

In [29]:
# cross validation results 
cvScores_av = [fold[2] for fold in cvScores]
cvScores_zygo =  [fold[0] for fold in cvScores]
cvScores_corr = [fold[1] for fold in cvScores]

avgScores_av = np.mean(cvScores_av,axis=0)
stdScores_av = np.std(cvScores_av)

avgScores_zygo = np.mean(cvScores_zygo,axis=0)
stdScores_zygo = np.std(cvScores_zygo)

avgScores_corr = np.mean(cvScores_corr,axis=0)
stdScores_corr = np.std(cvScores_corr)

print(f"Average KFold Cross Validation Score: {avgScores_av}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores_av}")
print("\n")

print(f"Average KFold Cross Validation Score Zygo: {avgScores_zygo}")
print(f"Standard Deviation KFold Cross Validation Score Zygo: {stdScores_zygo}")
print("\n")

print(f"Average KFold Cross Validation Score Corr: {avgScores_corr}")
print(f"Standard Deviation KFold Cross Validation Score Corr: {stdScores_corr}")
print("\n")

# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model.predict(X_test)

# Convert probabilities to class labels
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)

# true labels for corr and zygo 
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]
y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# calculate accuracy for each muscle group 
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)

# calculate f1 score for each muscle group 
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')

print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n -------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)

# average score
print("\n -------- Average --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)  

Average KFold Cross Validation Score: 86.3007378578186
Standard Deviation KFold Cross Validation Score: 0.1687016623717976


Average KFold Cross Validation Score Zygo: 86.53711915016174
Standard Deviation KFold Cross Validation Score Zygo: 0.3561473356657464


Average KFold Cross Validation Score Corr: 86.06435656547546
Standard Deviation KFold Cross Validation Score Corr: 0.2598113294699718


45/45 [==============================] - 0s 8ms/step
-------- Zygo --------
Training Accuracy: 0.9283963585434174
Test Accuracy: 0.8830532212885154
Training F1 Score: 0.9219597317126815
Test F1 Score: 0.8583119144096761

 -------- Corr --------
Training Accuracy: 0.9227941176470589
Test Accuracy: 0.8690476190476191
Training F1 Score: 0.9149332477907669
Test F1 Score: 0.8390924121765719

 -------- Average --------
Training Accuracy: 0.9255952380952381
Test Accuracy: 0.8760504201680672
Training F1 Score: 0.9184464897517242
Test F1 Score: 0.848702163293124


Single Head Duration 

In [11]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 100 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_dur = CNN_model_duration_multichan(input_shape, num_classes,feature_num)
    model_dur.compile(optimizer=Adam(learning_rate=0.0001, clipnorm=1.0),
    loss={
        'zygo_output': 'mae',
        'corr_output': 'mae'
    },
    metrics={
        'zygo_output': ['mae'],
        'corr_output': ['mae']
    }
)
    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model_dur.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]]), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_dur.evaluate(X_val,[y_val[:, 0], y_val[:, 1]])
    for name, value in zip(model_dur.metrics_names, scores):
        print(f"{name}: {value:.4f}")
    
    cvScores.append([scores[3],scores[4],( scores[3] + scores[4]) / 2 ]) 

    k += 1 
    

model_history = model_dur.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

In [19]:
model_history = model_dur.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

Epoch 1/100
179/179 [==============================] - 12s 64ms/step - loss: nan - zygo_output_loss: 4.0333 - corr_output_loss: 3.9536 - zygo_output_mae: 4.0333 - corr_output_mae: 3.9536 - val_loss: nan - val_zygo_output_loss: 4.1678 - val_corr_output_loss: 4.2360 - val_zygo_output_mae: 4.1678 - val_corr_output_mae: 4.2360
Epoch 2/100
179/179 [==============================] - 11s 60ms/step - loss: nan - zygo_output_loss: 4.0333 - corr_output_loss: 3.9536 - zygo_output_mae: 4.0333 - corr_output_mae: 3.9536 - val_loss: nan - val_zygo_output_loss: 4.1678 - val_corr_output_loss: 4.2360 - val_zygo_output_mae: 4.1678 - val_corr_output_mae: 4.2360
Epoch 3/100
179/179 [==============================] - 10s 54ms/step - loss: nan - zygo_output_loss: 4.0333 - corr_output_loss: 3.9537 - zygo_output_mae: 4.0333 - corr_output_mae: 3.9537 - val_loss: nan - val_zygo_output_loss: 4.1678 - val_corr_output_loss: 4.2360 - val_zygo_output_mae: 4.1678 - val_corr_output_mae: 4.2360
Epoch 4/100
179/179 [====

In [20]:
# cross validation results 
# cross validation results 
cvScores_av = [fold[2] for fold in cvScores]
cvScores_zygo =  [fold[0] for fold in cvScores]
cvScores_corr = [fold[1] for fold in cvScores]

avgScores_av = np.mean(cvScores_av,axis=0)
stdScores_av = np.std(cvScores_av)

avgScores_zygo = np.mean(cvScores_zygo,axis=0)
stdScores_zygo = np.std(cvScores_zygo)

avgScores_corr = np.mean(cvScores_corr,axis=0)
stdScores_corr = np.std(cvScores_corr)

print(f"Average KFold Cross Validation Score: {avgScores_av}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores_av}")
print("\n")

print(f"Average KFold Cross Validation Score Zygo: {avgScores_zygo}")
print(f"Standard Deviation KFold Cross Validation Score Zygo: {stdScores_zygo}")
print("\n")

print(f"Average KFold Cross Validation Score Corr: {avgScores_corr}")
print(f"Standard Deviation KFold Cross Validation Score Corr: {stdScores_corr}")
print("\n")

# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model_dur.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model_dur.predict(X_test)


baseline_zygo = np.mean(y_train_full[:,0])
mae_baseline_zygo = np.mean(np.abs(y_train_full[:,0] - baseline_zygo))

baseline_corr = np.mean(y_train_full[:,1])
mae_baseline_corr = np.mean(np.abs(y_train_full[:,1] - baseline_corr))
  
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,0], y_pred_train_zygo)   
mae_test_dur_zygo = mean_absolute_error(y_test[:,0], y_pred_test_zygo)  

mae_training_dur_corr = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo)   
mae_test_dur_corr= mean_absolute_error(y_test[:,1], y_pred_test_zygo)  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)
print("----------------------") 
print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)

Average KFold Cross Validation Score: 3.993370771408081
Standard Deviation KFold Cross Validation Score: 0.1536227158744175


Average KFold Cross Validation Score Zygo: 4.033257722854614
Standard Deviation KFold Cross Validation Score Zygo: 0.13629364964189003


Average KFold Cross Validation Score Corr: 3.953483819961548
Standard Deviation KFold Cross Validation Score Corr: 0.19102832870765313


45/45 [==============================] - 1s 12ms/step
Baseline MAE corr: 5.768655971313989
Training MAE corr : 3.9528214757677294
Test MAE corr: 4.235196597725111
----------------------
Baseline MAE zygo: 5.969496176708904
Training MAE zygo : 4.0325625750300045
Test MAE zygo: 4.1670333827079205


# Four Head 

In [5]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y_contr = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])


y_dur = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])


y = np.column_stack((y_contr, y_dur))
y = y[:, [0, 2, 1, 3]]

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [6]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_fourhead=[]
epoch_num = 10 
k = 1
from sklearn.preprocessing import StandardScaler

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X_train_full[train_index], X_train_full[test_index]
    y_train, y_val = y_train_full[train_index], y_train_full[test_index]

    model_fourhead = CNN_model_fourhead_multichan(input_shape, num_classes,feature_num)
    model_fourhead.compile(
    optimizer=Adam(learning_rate=0.0001, clipnorm=1.0),
    loss={
        'zygo_count_output': 'sparse_categorical_crossentropy',
        'corr_count_output': 'sparse_categorical_crossentropy',
        'zygo_duration_output': 'mae',
        'corr_duration_output': 'mae'
    },loss_weights={
            'zygo_count_output':    1.0,
            'corr_count_output':    1.0,
            'zygo_duration_output': 0.01,
            'corr_duration_output': 0.01
        },
    
    metrics={
        'zygo_count_output': ['accuracy'],
        'corr_count_output': ['accuracy'],
        'zygo_duration_output': ['mae'],
        'corr_duration_output': ['mae']
    }
)
    early_stop = EarlyStopping(monitor='val_loss',  patience=6, restore_best_weights=True)
    
    model_history_kfold = model_fourhead.fit(X_train, {
        'zygo_count_output': y_train[:, 0],
        'zygo_duration_output': y_train[:, 1],
        'corr_count_output': y_train[:, 2],
        'corr_duration_output': y_train[:, 3]
    }, epochs=epoch_num, validation_data=(
    X_val,
    {
        'zygo_count_output': y_val[:, 0],
        'zygo_duration_output': y_val[:, 1],
        'corr_count_output': y_val[:, 2],
        'corr_duration_output': y_val[:, 3]
    }
), callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)

    scores = model_fourhead.evaluate(X_val,[y_val[:, 0], y_val[:, 1]])
    for name, value in zip(model_fourhead.metrics_names, scores):
        print(f"{name}: {value:.4f}")
    cvScores_fourhead.append(scores)

    k += 1 
    
model_history_fourhead = model_fourhead.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]]), callbacks=[early_stop])

Fold: 1 ==================================================================


2026-05-12 20:15:59.529293: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-05-12 20:15:59.529919: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-05-12 20:15:59.533239: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-05-12 20:15:59.533705: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-12 20:15:59.534282: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/20


2026-05-12 20:16:02.190842: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-05-12 20:16:02.391565: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


143/143 [==============================] - 20s 106ms/step - loss: 29.7297 - zygo_count_output_loss: 5.9523 - zygo_duration_output_loss: 9.4146 - corr_count_output_loss: 6.0287 - corr_duration_output_loss: 7.9914 - zygo_count_output_accuracy: 0.2793 - zygo_duration_output_mae: 9.4146 - corr_count_output_accuracy: 0.2445 - corr_duration_output_mae: 7.9914 - val_loss: 20.8291 - val_zygo_count_output_loss: 8.9874 - val_zygo_duration_output_loss: 1.4754 - val_corr_count_output_loss: 8.8830 - val_corr_duration_output_loss: 1.1390 - val_zygo_count_output_accuracy: 0.0639 - val_zygo_duration_output_mae: 1.4754 - val_corr_count_output_accuracy: 0.0761 - val_corr_duration_output_mae: 1.1390
Epoch 2/20
143/143 [==============================] - 13s 89ms/step - loss: 30.7145 - zygo_count_output_loss: 2.6519 - zygo_duration_output_loss: 13.6436 - corr_count_output_loss: 2.5223 - corr_duration_output_loss: 11.5516 - zygo_count_output_accuracy: 0.6616 - zygo_duration_output_mae: 13.6436 - corr_count_

...

143/143 [==============================] - 18s 102ms/step - loss: 31.5958 - zygo_count_output_loss: 6.0868 - zygo_duration_output_loss: 9.6270 - corr_count_output_loss: 6.4610 - corr_duration_output_loss: 9.0775 - zygo_count_output_accuracy: 0.1315 - zygo_duration_output_mae: 9.6270 - corr_count_output_accuracy: 0.1554 - corr_duration_output_mae: 9.0775 - val_loss: 21.6043 - val_zygo_count_output_loss: 9.2945 - val_zygo_duration_output_loss: 1.5644 - val_corr_count_output_loss: 9.1536 - val_corr_duration_output_loss: 1.2465 - val_zygo_count_output_accuracy: 0.0298 - val_zygo_duration_output_mae: 1.5644 - val_corr_count_output_accuracy: 0.0228 - val_corr_duration_output_mae: 1.2465
Epoch 2/20
143/143 [==============================] - 12s 86ms/step - loss: 37.7303 - zygo_count_output_loss: 2.3263 - zygo_duration_output_loss: 17.2281 - corr_count_output_loss: 2.5841 - corr_duration_output_loss: 15.2449 - zygo_count_output_accuracy: 0.5928 - zygo_duration_output_mae: 17.2281 - corr_count_

2026-05-12 20:29:16.620630: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


143/143 [==============================] - 17s 95ms/step - loss: 32.9160 - zygo_count_output_loss: 6.9960 - zygo_duration_output_loss: 11.1433 - corr_count_output_loss: 5.8778 - corr_duration_output_loss: 8.5552 - zygo_count_output_accuracy: 0.0578 - zygo_duration_output_mae: 11.1433 - corr_count_output_accuracy: 0.2361 - corr_duration_output_mae: 8.5552 - val_loss: 20.6098 - val_zygo_count_output_loss: 8.4688 - val_zygo_duration_output_loss: 2.0235 - val_corr_count_output_loss: 8.6030 - val_corr_duration_output_loss: 1.1692 - val_zygo_count_output_accuracy: 0.0744 - val_zygo_duration_output_mae: 2.0235 - val_corr_count_output_accuracy: 0.1086 - val_corr_duration_output_mae: 1.1692
Epoch 2/20
143/143 [==============================] - 14s 98ms/step - loss: 35.5998 - zygo_count_output_loss: 3.0092 - zygo_duration_output_loss: 17.0813 - corr_count_output_loss: 2.5301 - corr_duration_output_loss: 12.6327 - zygo_count_output_accuracy: 0.5249 - zygo_duration_output_mae: 17.0813 - corr_count

2026-05-12 20:33:15.006900: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp.


143/143 [==============================] - 17s 95ms/step - loss: 28.6464 - zygo_count_output_loss: 6.3104 - zygo_duration_output_loss: 7.8903 - corr_count_output_loss: 5.8420 - corr_duration_output_loss: 8.2601 - zygo_count_output_accuracy: 0.1853 - zygo_duration_output_mae: 7.8903 - corr_count_output_accuracy: 0.2319 - corr_duration_output_mae: 8.2601 - val_loss: 20.5425 - val_zygo_count_output_loss: 8.5842 - val_zygo_duration_output_loss: 1.3681 - val_corr_count_output_loss: 8.6936 - val_corr_duration_output_loss: 1.5509 - val_zygo_count_output_accuracy: 0.0587 - val_zygo_duration_output_mae: 1.3681 - val_corr_count_output_accuracy: 0.0613 - val_corr_duration_output_mae: 1.5509
Epoch 2/20
143/143 [==============================] - 12s 84ms/step - loss: 33.8481 - zygo_count_output_loss: 2.8003 - zygo_duration_output_loss: 13.8603 - corr_count_output_loss: 2.5243 - corr_duration_output_loss: 14.3163 - zygo_count_output_accuracy: 0.6247 - zygo_duration_output_mae: 13.8603 - corr_count_o

In [1]:

# Cross-validation results
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")


y_pred_train = model_fourhead.predict(X_train_full)
y_pred_test = model_fourhead.predict(X_test)

y_pred_train_zygo_count, y_pred_train_zygo_dur, y_pred_train_corr_count, y_pred_train_corr_dur = y_pred_train
y_pred_test_zygo_count, y_pred_test_zygo_dur, y_pred_test_corr_count, y_pred_test_corr_dur = y_pred_test


# baselines
baseline_zygo = np.mean(y_train_full[:,1])
baseline_corr = np.mean(y_train_full[:,3])

mae_baseline_zygo = np.mean(np.abs(y_train_full[:,1] - baseline_zygo))
mae_baseline_corr = np.mean(np.abs(y_train_full[:,3] - baseline_corr))

# MAE
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo_dur)
mae_test_dur_zygo = mean_absolute_error(y_test[:,1], y_pred_test_zygo_dur)

mae_training_dur_corr = mean_absolute_error(y_train_full[:,3], y_pred_train_corr_dur)
mae_test_dur_corr = mean_absolute_error(y_test[:,3], y_pred_test_corr_dur)

print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)

print("----------------------") 

print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)



y_pred_train_zygo_count = np.argmax(y_pred_train_zygo_count, axis=1)
y_pred_train_corr_count = np.argmax(y_pred_train_corr_count, axis=1)

y_pred_test_zygo_count = np.argmax(y_pred_test_zygo_count, axis=1)
y_pred_test_corr_count = np.argmax(y_pred_test_corr_count, axis=1)

# true labels
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]

y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# accuracy
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo_count)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr_count)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo_count)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr_count)

# f1
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo_count, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr_count, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo_count, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr_count, average='weighted')



print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n-------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)


print("\n-------- Average (Counts) --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)

print("\n-------- Average (Duration MAE) --------")
print("Training MAE:", (mae_training_dur_zygo + mae_training_dur_corr) / 2)
print("Test MAE:", (mae_test_dur_zygo + mae_test_dur_corr) / 2)

NameError: name 'np' is not defined

## Segmentation

In [ ]:
def comparator(learner, instructor):
    if len(learner) != len(instructor):
        raise AssertionError("Layer count mismatch")
    for a, b in zip(learner, instructor):
        if tuple(a) != tuple(b):
            print(colored("Test failed", attrs=['bold']))
            raise AssertionError("Error in test")
    print(colored("All tests passed!", "green"))

def summary(model):
    result = []
    for layer in model.layers:
        output_shape = getattr(layer.output, 'shape', None)
        params = layer.count_params() if hasattr(layer, 'count_params') else 0
        result.append([layer.__class__.__name__, output_shape, params])
    return result
